# FRUTINOVELAS · Montaje con audio
CPU; sin modelos generativos. Conserva el audio maestro y ajusta imágenes o clips a cada toma (máximo cinco minutos). Un clip corto mantiene su último fotograma hasta el siguiente corte.

Ejecuta las tres celdas en orden. El entorno es temporal: descarga el resultado en Studio antes de cerrar Colab. No subas esta libreta ejecutada con la clave visible.


In [ ]:
# @title 1. Preparar montador (CPU, sin modelos)
import sys, subprocess, shutil, urllib.request
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp>=3.10,<4", "Pillow>=10.4"], check=True)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
work = Path("/content/frutinovelas-montaje")
work.mkdir(parents=True, exist_ok=True)
binary = work / "cloudflared"
if not binary.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
    binary.chmod(0o700)
print("Montador preparado.")


In [ ]:
# @title 2. Instalar servidor incluido
sources = {'narrated_gateway.py': '"""Authenticated, server-side FFmpeg assembly for RunPod and Colab.\n\nOnly pre-uploaded, hash-addressed images/audio or files confined to ComfyUI\'s\noutput directory can be used. No client-supplied paths or shell commands.\n"""\nfrom __future__ import annotations\n\nimport asyncio\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport subprocess\nimport uuid\nfrom pathlib import Path\n\nfrom aiohttp import web\nfrom PIL import Image, UnidentifiedImageError\n\nMAX_FILE_BYTES = 64 * 1024 * 1024\nMAX_SECONDS = 300\nMAX_SHOTS = 90\nCHUNK = 1024 * 1024\nSHA = re.compile(r"^[0-9a-f]{64}$")\nCLIP_NAME = re.compile(r"^[A-Za-z0-9_. -]{1,200}\\.mp4$", re.I)\nCLIP_FOLDER = re.compile(r"^[A-Za-z0-9_-]+(?:/[A-Za-z0-9_-]+)*$")\nMOTIONS = frozenset(("zoom_in", "zoom_out", "pan_left", "pan_right", "still"))\nEXTENSIONS = {\n    "image": (".png", ".jpg", ".webp"),\n    "audio": (".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".webm"),\n    "video": (".mp4",),\n}\n\n\nclass AssemblyError(ValueError):\n    pass\n\n\ndef run_ffmpeg(args: list[str], timeout: int = 600) -> None:\n    try:\n        process = subprocess.run(args, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,\n                                 stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        raise AssemblyError("No se pudo completar el montaje remoto con FFmpeg.") from exc\n    if process.returncode:\n        raise AssemblyError("FFmpeg falló en una toma o al unirlas; revisa las imágenes y clips.")\n\n\ndef probe(path: Path) -> dict:\n    binary = shutil.which("ffprobe")\n    if not binary:\n        raise AssemblyError("Falta FFprobe en el servidor de ComfyUI.")\n    try:\n        process = subprocess.run(\n            [binary, "-v", "error", "-show_entries", "format=duration:stream=codec_type",\n             "-of", "json", str(path)],\n            stdin=subprocess.DEVNULL, capture_output=True, timeout=25, check=True,\n        )\n        result = json.loads(process.stdout)\n        return result\n    except (OSError, subprocess.CalledProcessError, subprocess.TimeoutExpired, ValueError) as exc:\n        raise AssemblyError("El servidor no pudo validar un audio o clip.") from exc\n\n\ndef audio_duration(path: Path) -> float:\n    info = probe(path)\n    streams = info.get("streams", [])\n    if len([s for s in streams if s.get("codec_type") == "audio"]) != 1 or any(\n        s.get("codec_type") == "video" for s in streams\n    ):\n        raise AssemblyError("La narración debe contener una sola pista de audio y ningún video.")\n    try:\n        duration = float(info["format"]["duration"])\n    except (KeyError, TypeError, ValueError) as exc:\n        raise AssemblyError("El audio no tiene una duración reconocible.") from exc\n    if not math.isfinite(duration) or not 0 < duration <= MAX_SECONDS:\n        raise AssemblyError("El audio debe durar hasta cinco minutos.")\n    return duration\n\n\ndef image_extension(path: Path) -> str:\n    try:\n        with Image.open(path) as picture:\n            picture.verify()\n        with Image.open(path) as picture:\n            fmt = picture.format\n            width, height = picture.size\n    except (OSError, UnidentifiedImageError, Image.DecompressionBombError) as exc:\n        raise AssemblyError("La imagen subida no es válida.") from exc\n    if not 16 <= width <= 8192 or not 16 <= height <= 8192:\n        raise AssemblyError("La imagen supera los límites permitidos.")\n    return {"PNG": ".png", "JPEG": ".jpg", "WEBP": ".webp"}.get(fmt) or _invalid_image()\n\n\ndef _invalid_image():\n    raise AssemblyError("Solo se permiten imágenes PNG, JPEG o WebP.")\n\n\ndef audio_extension(path: Path) -> str:\n    # Require actual FFprobe demuxer detection; untrusted filename is ignored.\n    audio_duration(path)\n    args = [shutil.which("ffprobe"), "-v", "error", "-show_entries", "format=format_name",\n            "-of", "json", str(path)]\n    try:\n        raw = subprocess.run(args, stdin=subprocess.DEVNULL, capture_output=True, timeout=20, check=True)\n        formats = set(json.loads(raw.stdout)["format"]["format_name"].split(","))\n    except (OSError, KeyError, ValueError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:\n        raise AssemblyError("El formato del audio es desconocido.") from exc\n    for fmt, extension in (("wav", ".wav"), ("mp3", ".mp3"), ("mov", ".m4a"),\n                           ("aac", ".aac"), ("flac", ".flac"), ("ogg", ".ogg"),\n                           ("matroska", ".webm")):\n        if fmt in formats:\n            return extension\n    raise AssemblyError("Formato de audio no admitido.")\n\n\ndef video_extension(path: Path) -> str:\n    info = probe(path)\n    streams = info.get("streams", [])\n    try:\n        duration = float(info["format"]["duration"])\n    except (KeyError, TypeError, ValueError) as exc:\n        raise AssemblyError("El clip no tiene una duración válida.") from exc\n    if not any(s.get("codec_type") == "video" for s in streams) or not math.isfinite(duration) or not 0 < duration <= MAX_SECONDS:\n        raise AssemblyError("El recurso no contiene un clip válido de hasta cinco minutos.")\n    return ".mp4"\n\n\ndef motion_filter(motion: str, seconds: float, width: int, height: int) -> str:\n    frames = max(1, round(seconds * 24))\n    zoom = {\n        "zoom_in": f"1+0.10*on/{frames}",\n        "zoom_out": f"1.10-0.10*on/{frames}",\n        "pan_left": "1.10",\n        "pan_right": "1.10",\n        "still": "1",\n    }[motion]\n    x = "iw/2-(iw/zoom/2)"\n    if motion == "pan_left":\n        x = f"(iw-iw/zoom)*(1-on/{frames})"\n    elif motion == "pan_right":\n        x = f"(iw-iw/zoom)*on/{frames}"\n    return (f"scale={width*2}:{height*2}:force_original_aspect_ratio=increase,"\n            f"crop={width*2}:{height*2},zoompan=z=\'{zoom}\':x=\'{x}\':"\n            f"y=\'ih/2-ih/zoom/2\':d={frames}:s={width}x{height}:fps=24,"\n            "format=yuv420p")\n\n\nclass NarrationStore:\n    def __init__(self, root: Path, comfy_output: Path | None = None):\n        self.root = root.resolve()\n        self.assets = self.root / "assets"\n        self.jobs = self.root / "jobs"\n        self.output = self.root / "exports"\n        self.comfy_output = comfy_output.resolve() if comfy_output else None\n        for folder in (self.assets, self.jobs, self.output):\n            folder.mkdir(parents=True, exist_ok=True)\n        identity_path = self.root / "runtime-id"\n        try:\n            with identity_path.open("x", encoding="ascii") as identity:\n                identity.write(str(uuid.uuid4()))\n        except FileExistsError:\n            pass\n        self.runtime_id = str(uuid.UUID(identity_path.read_text(encoding="ascii").strip()))\n        self.lock = asyncio.Lock()\n        self.tasks: set[asyncio.Task] = set()\n\n    def asset(self, sha: str, kind: str) -> Path:\n        if not isinstance(sha, str) or not SHA.fullmatch(sha) or kind not in EXTENSIONS:\n            raise AssemblyError("Referencia de recurso inválida.")\n        for extension in EXTENSIONS[kind]:\n            path = self.assets / (sha + extension)\n            if path.is_file():\n                return path\n        raise AssemblyError("El recurso no está disponible en el servidor remoto.")\n\n    def clip(self, filename: str, folder: str) -> Path:\n        if not self.comfy_output:\n            raise AssemblyError("No se configuró el directorio de salida de ComfyUI.")\n        if not isinstance(filename, str) or not CLIP_NAME.fullmatch(filename) or ".." in filename:\n            raise AssemblyError("Nombre de clip remoto no permitido.")\n        if not isinstance(folder, str) or (folder and (not CLIP_FOLDER.fullmatch(folder) or ".." in folder)):\n            raise AssemblyError("Subcarpeta de clip inválida.")\n        base = self.comfy_output\n        path = (base / folder / filename).resolve()\n        if base not in path.parents or not path.is_file() or path.is_symlink():\n            raise AssemblyError("El clip no existe en la salida de ComfyUI.")\n        return path\n\n    def filename(self, job_id: str, suffix: str = ".json") -> Path:\n        try:\n            normalized = str(uuid.UUID(job_id))\n        except (TypeError, ValueError, AttributeError) as exc:\n            raise AssemblyError("ID de montaje inválido.") from exc\n        if normalized != job_id:\n            raise AssemblyError("El ID debe ser un UUID normalizado.")\n        return self.jobs / (job_id + suffix)\n\n    def read(self, job_id: str) -> dict | None:\n        path = self.filename(job_id)\n        if not path.is_file():\n            return None\n        entry = json.loads(path.read_text(encoding="utf-8"))\n        if entry["state"] in ("queued", "rendering"):\n            # As another worker might still be generating, this branch is used only\n            # at gateway startup, not on every request.\n            pass\n        return entry\n\n    def save(self, row: dict) -> None:\n        path = self.filename(row["id"])\n        tmp = path.with_suffix(".tmp")\n        tmp.write_text(json.dumps(row, ensure_ascii=False), encoding="utf-8")\n        os.replace(tmp, path)\n\n    def recover(self):\n        for file in self.jobs.glob("*.json"):\n            try:\n                row = json.loads(file.read_text(encoding="utf-8"))\n                if row.get("state") in ("queued", "rendering"):\n                    row["state"] = "failed"\n                    row["error"] = "El servidor remoto se reinició; vuelve a exportar con otro ID."\n                    self.save(row)\n            except (OSError, ValueError, KeyError):\n                continue\n\n    def validate(self, data: dict) -> tuple[Path, list[dict], float]:\n        if not isinstance(data, dict):\n            raise AssemblyError("El manifiesto no es válido.")\n        job_id = data.get("id")\n        self.filename(job_id)\n        audio = self.asset(data.get("audio_id"), "audio")\n        duration = audio_duration(audio)\n        if data.get("aspect_ratio") not in ("9:16", "16:9"):\n            raise AssemblyError("Formato de video no admitido.")\n        shots = data.get("shots")\n        if not isinstance(shots, list) or not 1 <= len(shots) <= MAX_SHOTS:\n            raise AssemblyError("El montaje necesita entre 1 y 90 tomas.")\n        previous = 0.0\n        for i, shot in enumerate(shots):\n            if not isinstance(shot, dict):\n                raise AssemblyError("Una toma no tiene formato válido.")\n            start, end, motion = shot.get("start_seconds"), shot.get("end_seconds"), shot.get("motion", "zoom_in")\n            if not all(isinstance(v, (float, int)) and not isinstance(v, bool) and math.isfinite(v) for v in (start, end)):\n                raise AssemblyError(f"Marcas de tiempo inválidas en la toma {i+1}.")\n            if start < 0 or end-start < 0.05 or end > duration+0.06 or abs(start-previous) > 0.06:\n                raise AssemblyError(f"La toma {i+1} no coincide con la cronología del audio.")\n            if motion not in MOTIONS:\n                raise AssemblyError("Movimiento de cámara no permitido.")\n            if shot.get("clip_id"):\n                if shot.get("clip_filename") or shot.get("image_id"):\n                    raise AssemblyError("Una toma no puede mezclar imagen y clip.")\n                video_extension(self.asset(shot["clip_id"], "video"))\n            elif shot.get("clip_filename"):\n                path = self.clip(shot["clip_filename"], shot.get("clip_subfolder", ""))\n                if not any(stream.get("codec_type") == "video" for stream in probe(path).get("streams", [])):\n                    raise AssemblyError("El archivo H3 no contiene video.")\n            else:\n                self.asset(shot.get("image_id"), "image")\n            previous = end\n        if abs(previous-duration) > 0.06:\n            raise AssemblyError("Faltan tomas para cubrir toda la narración.")\n        return audio, shots, duration\n\n    def assemble(self, data: dict) -> None:\n        job_id = data["id"]\n        row = self.read(job_id)\n        row["state"] = "rendering"\n        self.save(row)\n        tmpdir = self.root / ("working-" + job_id)\n        tmpdir.mkdir(mode=0o700, exist_ok=True)\n        try:\n            ffmpeg = shutil.which("ffmpeg")\n            if not ffmpeg:\n                raise AssemblyError("Instala FFmpeg en RunPod o Colab.")\n            audio, shots, duration = self.validate(data)\n            width, height = (576, 1024) if data["aspect_ratio"] == "9:16" else (1024, 576)\n            clips: list[Path] = []\n            for i, shot in enumerate(shots):\n                seconds = float(shot["end_seconds"]) - float(shot["start_seconds"])\n                frames = max(1, round(float(shot["end_seconds"]) * 24) - round(float(shot["start_seconds"]) * 24))\n                clip = tmpdir / f"{i:03}.mp4"\n                args = [ffmpeg, "-y", "-nostdin", "-v", "error"]\n                if shot.get("clip_id") or shot.get("clip_filename"):\n                    path = (self.asset(shot["clip_id"], "video") if shot.get("clip_id") else\n                            self.clip(shot["clip_filename"], shot.get("clip_subfolder", "")))\n                    args += ["-i", str(path), "-vf", f"scale={width}:{height}:force_original_aspect_ratio=increase,"\n                             f"crop={width}:{height},fps=24,tpad=stop_mode=clone:stop_duration={seconds:.5f},"\n                             "format=yuv420p"]\n                else:\n                    path = self.asset(shot["image_id"], "image")\n                    args += ["-loop", "1", "-framerate", "24", "-i", str(path),\n                             "-vf", motion_filter(shot["motion"], seconds, width, height)]\n                args += ["-frames:v", str(frames), "-an", "-c:v", "libx264", "-preset", "veryfast",\n                         "-crf", "21", "-pix_fmt", "yuv420p", str(clip)]\n                run_ffmpeg(args, timeout=180)\n                clips.append(clip)\n                row["progress"] = round((i + 1) * 90 / len(shots))\n                self.save(row)\n            manifest = tmpdir / "concat.txt"\n            # Names are generated by our server, not by user input.\n            manifest.write_text("".join(f"file \'{clip.as_posix()}\'\\n" for clip in clips), encoding="utf-8")\n            partial = tmpdir / "complete.mp4"\n            run_ffmpeg([ffmpeg, "-y", "-nostdin", "-v", "error", "-f", "concat", "-safe", "0",\n                        "-i", str(manifest), "-i", str(audio),\n                        "-map", "0:v:0", "-map", "1:a:0",\n                        "-vf", "fps=24,tpad=stop_mode=clone:stop_duration=1,format=yuv420p",\n                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",\n                        "-c:a", "aac", "-b:a", "192k", "-t", f"{duration:.5f}",\n                        "-movflags", "+faststart", str(partial)], timeout=900)\n            output = self.output / (job_id + ".mp4")\n            os.replace(partial, output)\n            row["state"] = "completed"\n            row["progress"] = 100\n            self.save(row)\n        except Exception as exc:\n            row["state"] = "failed"\n            row["error"] = str(exc) if isinstance(exc, AssemblyError) else "Error del montaje remoto; consulta el log del gateway."\n            self.save(row)\n        finally:\n            shutil.rmtree(tmpdir, ignore_errors=True)\n\n    async def submit(self, data: dict) -> dict:\n        # Validate the job *before* reserving the ID.\n        await asyncio.to_thread(self.validate, data)\n        signature = hashlib.sha256(json.dumps(data, sort_keys=True).encode()).hexdigest()\n        async with self.lock:\n            existing = self.read(data["id"])\n            if existing is not None:\n                if existing.get("signature") != signature:\n                    raise web.HTTPConflict(text="Ese ID pertenece a otro montaje.")\n                return self.public(existing)\n            row = {"id": data["id"], "signature": signature, "manifest": data,\n                   "state": "queued", "progress": 0, "error": ""}\n            self.save(row)\n            task = asyncio.create_task(asyncio.to_thread(self.assemble, data))\n            self.tasks.add(task)\n            task.add_done_callback(self.tasks.discard)\n            return self.public(row)\n\n    async def remove_export(self, job_id: str) -> dict:\n        """Remove exactly this remote completed MP4 and its job record."""\n        async with self.lock:\n            row = self.read(job_id)\n            if row is None:\n                return {"deleted": False, "bytes_freed": 0}\n            if row["state"] in ("queued", "rendering"):\n                raise web.HTTPConflict(text="No se puede eliminar una exportación en curso.")\n            target = self.output / (job_id + ".mp4")\n            size = target.stat().st_size if target.is_file() and not target.is_symlink() else 0\n            if target.is_symlink():\n                raise web.HTTPConflict(text="El archivo de salida no es seguro.")\n            target.unlink(missing_ok=True)\n            self.filename(job_id).unlink(missing_ok=True)\n            return {"deleted": True, "bytes_freed": size}\n\n    async def remove_staged_asset(self, sha: str, kind: str) -> dict:\n        """Reclaim a staged copy, never an input of an active montage."""\n        async with self.lock:\n            if not isinstance(sha, str) or not SHA.fullmatch(sha) or kind not in EXTENSIONS:\n                raise AssemblyError("El recurso remoto solicitado no es válido.")\n            for job_file in self.jobs.glob("*.json"):\n                try:\n                    job = json.loads(job_file.read_text(encoding="utf-8"))\n                except (OSError, ValueError):\n                    raise web.HTTPConflict(text="No es seguro borrar mientras hay trabajos ilegibles.")\n                if job.get("state") not in ("queued", "rendering"):\n                    continue\n                manifest = job.get("manifest") or {}\n                used = {manifest.get("audio_id")} | {\n                    resource for shot in manifest.get("shots", []) if isinstance(shot, dict)\n                    for resource in (shot.get("image_id"), shot.get("clip_id"))\n                }\n                if sha in used or not manifest:\n                    raise web.HTTPConflict(text="La imagen o el audio se utiliza en un montaje activo.")\n            paths = [self.assets / (sha + ext) for ext in EXTENSIONS[kind]]\n            freed = 0\n            for path in paths:\n                if path.is_symlink():\n                    raise web.HTTPConflict(text="Recurso no seguro para borrar.")\n                if path.is_file():\n                    freed += path.stat().st_size\n                    path.unlink()\n            return {"deleted": freed > 0, "bytes_freed": freed}\n\n    async def remove_generation_file(self, job_id: str, kind: str) -> dict:\n        """Delete generated ComfyUI output(s) with an exact UUID-owned prefix.\n\n        Never traverse folders or delete unrelated ComfyUI outputs.\n        """\n        self.filename(job_id)  # validates normalized UUID\n        if kind not in ("image", "video"):\n            raise AssemblyError("Tipo de generación inválido.")\n        if not self.comfy_output:\n            raise AssemblyError("Falta la carpeta de salida de ComfyUI.")\n        folder = self.comfy_output / "FRUTINOVELAS"\n        if not folder.is_dir() or folder.is_symlink():\n            return {"deleted": False, "bytes_freed": 0}\n        prefix = (("IMAGE-" if kind == "image" else "") + job_id + "_")\n        allowed = {".png", ".jpg", ".webp"} if kind == "image" else {".mp4", ".webm"}\n        # Avoid deleting in-progress ComfyUI jobs: require an existing, terminal\n        # history record before removing any generated output.\n        # Studio also enforces this state before calling this endpoint.\n        async with self.lock:\n            files = [f for f in folder.iterdir() if f.name.startswith(prefix) and f.suffix.lower() in allowed]\n            if any(f.is_symlink() or not f.is_file() or f.resolve().parent != folder.resolve() for f in files):\n                raise web.HTTPConflict(text="Hay salidas no seguras: no se borró nada.")\n            freed = 0\n            for file in files:\n                freed += file.stat().st_size\n                file.unlink()\n            return {"deleted": bool(files), "files_deleted": len(files), "bytes_freed": freed}\n\n    @staticmethod\n    def public(row: dict) -> dict:\n        return {name: row[name] for name in ("id", "state", "progress", "error")}\n\n\ndef register_narration_routes(app: web.Application, root: Path, comfy_output: Path | None = None) -> None:\n    store = NarrationStore(root, comfy_output)\n    store.recover()\n\n    async def capabilities(_request):\n        return web.json_response({"supported": True, "engine": "remote-ffmpeg", "max_seconds": MAX_SECONDS,\n                                  "max_shots": MAX_SHOTS, "uploaded_video": True,\n                                  "runtime_id": store.runtime_id, "max_asset_bytes": MAX_FILE_BYTES,\n                                  "ffmpeg": bool(shutil.which("ffmpeg"))})\n\n    async def upload(request):\n        reader = await request.multipart()\n        kind: str | None = None\n        temp = store.root / ("upload-" + uuid.uuid4().hex)\n        sha = hashlib.sha256()\n        size = 0\n        try:\n            async for part in reader:\n                if part.name == "kind":\n                    kind = (await part.text())[:20]\n                elif part.name == "file":\n                    with temp.open("wb") as stream:\n                        while chunk := await part.read_chunk(size=CHUNK):\n                            size += len(chunk)\n                            if size > MAX_FILE_BYTES:\n                                raise web.HTTPRequestEntityTooLarge(max_size=MAX_FILE_BYTES, actual_size=size)\n                            sha.update(chunk)\n                            stream.write(chunk)\n            if kind not in EXTENSIONS or size == 0:\n                raise AssemblyError("Archivo o tipo no válido.")\n            validator = {"image": image_extension, "audio": audio_extension, "video": video_extension}[kind]\n            extension = await asyncio.to_thread(validator, temp)\n            dest = store.assets / (sha.hexdigest() + extension)\n            os.replace(temp, dest)\n            return web.json_response({"id": sha.hexdigest(), "kind": kind, "size": size})\n        except AssemblyError as exc:\n            raise web.HTTPUnprocessableEntity(text=str(exc)) from exc\n        finally:\n            temp.unlink(missing_ok=True)\n\n    async def create(request):\n        try:\n            data = await request.json()\n            return web.json_response(await store.submit(data), status=202)\n        except (AssemblyError, KeyError, TypeError, ValueError) as exc:\n            raise web.HTTPUnprocessableEntity(text=str(exc)) from exc\n\n    async def status(request):\n        try:\n            row = store.read(request.match_info["id"])\n        except AssemblyError:\n            raise web.HTTPNotFound()\n        if not row:\n            raise web.HTTPNotFound()\n        return web.json_response(store.public(row))\n\n    async def download(request):\n        job_id = request.match_info["id"]\n        try:\n            row = store.read(job_id)\n        except AssemblyError:\n            raise web.HTTPNotFound()\n        if not row or row["state"] != "completed":\n            raise web.HTTPNotFound()\n        path = store.output / (job_id + ".mp4")\n        if not path.is_file():\n            raise web.HTTPNotFound()\n        return web.FileResponse(path, headers={"Content-Type": "video/mp4"})\n\n    async def delete_export(request):\n        try:\n            return web.json_response(await store.remove_export(request.match_info["id"]))\n        except AssemblyError as exc:\n            raise web.HTTPUnprocessableEntity(text=str(exc)) from exc\n\n    async def delete_asset(request):\n        try:\n            return web.json_response(await store.remove_staged_asset(\n                request.match_info["sha"], request.query.get("kind", "")))\n        except AssemblyError as exc:\n            raise web.HTTPUnprocessableEntity(text=str(exc)) from exc\n\n    async def delete_generation(request):\n        try:\n            return web.json_response(await store.remove_generation_file(\n                request.match_info["id"], request.query.get("kind", "")))\n        except AssemblyError as exc:\n            raise web.HTTPUnprocessableEntity(text=str(exc)) from exc\n\n    app.router.add_get("/frutinovelas/narration/capabilities", capabilities)\n    app.router.add_post("/frutinovelas/narration/assets", upload)\n    app.router.add_post("/frutinovelas/narration/jobs", create)\n    app.router.add_get("/frutinovelas/narration/jobs/{id}", status)\n    app.router.add_get("/frutinovelas/narration/jobs/{id}/file", download)\n    app.router.add_delete("/frutinovelas/narration/jobs/{id}", delete_export)\n    app.router.add_delete("/frutinovelas/narration/assets/{sha}", delete_asset)\n    app.router.add_delete("/frutinovelas/narration/generated/{id}", delete_generation)\n', 'narrated_server.py': '"""Standalone narration assembly gateway, without ComfyUI or GPU inference."""\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport secrets\nfrom pathlib import Path\n\nfrom aiohttp import web\n\ntry:\n    from .narrated_gateway import register_narration_routes\nexcept ImportError:\n    from narrated_gateway import register_narration_routes\n\n\ndef create_app(token: str, root: Path) -> web.Application:\n    if len(token) < 24:\n        raise ValueError("Usa un token de al menos 24 caracteres.")\n\n    @web.middleware\n    async def authorize(request, handler):\n        if not secrets.compare_digest(request.headers.get("Authorization", ""), "Bearer " + token):\n            raise web.HTTPUnauthorized(text="Autenticación requerida.")\n        return await handler(request)\n\n    app = web.Application(middlewares=[authorize], client_max_size=70 * 1024**2)\n    register_narration_routes(app, root)\n    return app\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description="Montaje remoto de imágenes, clips y audio; no requiere GPU.")\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--port", type=int, default=8199)\n    args = parser.parse_args(argv)\n    token = os.environ.get("FRUTINOVELAS_NARRATION_TOKEN", "")\n    web.run_app(create_app(token, args.root), host="127.0.0.1", port=args.port, access_log=None)\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in sources.items():
    (work / name).write_text(source, encoding="utf-8")
print("Servidor instalado.")


In [ ]:
# @title 3. Conectar con Studio
import os, secrets, time, re, urllib.request, json
if "montage_server" in globals() and montage_server.poll() is None:
    raise RuntimeError("Ya hay un montador activo. Usa su conexión o reinicia el entorno antes de arrancar otro.")
montage_token = secrets.token_urlsafe(32)
environment = {**os.environ, "FRUTINOVELAS_NARRATION_TOKEN": montage_token}
server_log = (work / "server.log").open("w")
montage_server = subprocess.Popen([sys.executable, str(work / "narrated_server.py"), "--root", str(work / "data"), "--port", "8199"], env=environment, stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        req = urllib.request.Request("http://127.0.0.1:8199/frutinovelas/narration/capabilities", headers={"Authorization": "Bearer " + montage_token})
        capabilities = json.load(urllib.request.urlopen(req, timeout=2))
        if not capabilities.get("ffmpeg"):
            raise RuntimeError("FFmpeg no está disponible.")
        break
    except (OSError, ValueError):
        if montage_server.poll() is not None:
            raise RuntimeError("El montador no arrancó. Consulta /content/frutinovelas-montaje/server.log")
        time.sleep(.5)
else:
    raise RuntimeError("El montador no respondió a tiempo.")
tunnel_log = work / "tunnel.log"
tunnel_output = tunnel_log.open("w")
montage_tunnel = subprocess.Popen([str(binary), "tunnel", "--url", "http://127.0.0.1:8199", "--no-autoupdate"], stdout=tunnel_output, stderr=subprocess.STDOUT)
for _ in range(80):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text())
    if found:
        print("En Studio → Videos narrados → Conectar Colab para unir el video")
        print("URL:", found.group(0))
        print("Clave:", montage_token)
        print("Conserva esta clave en privado. Mantén Colab abierto; no requiere streaming SSE.")
        break
    time.sleep(.5)
else:
    raise RuntimeError("No apareció la URL del túnel. Consulta tunnel.log y vuelve a iniciar el entorno.")
